In [0]:
import requests

api_url = "https://api.jolpi.ca/ergast/f1/current/last/results.json"

response = requests.get(api_url, timeout=30)

print("HTTP status:", response.status_code)

response.raise_for_status()

payload = response.json()

print("Top-level JSON keys:", payload.keys())

In [0]:
race = payload["MRData"]["RaceTable"]["Races"][0]

print("Race:", race["raceName"])
print("Season:", race["season"])
print("Round:", race["round"])
print("Date:", race["date"])
print("Circuit:", race["Circuit"]["circuitName"])

winner = race["Results"][0]

print("\nWinner:", winner["Driver"]["givenName"], winner["Driver"]["familyName"])
print("Winning constructor:", winner["Constructor"]["name"])
print("Points:", winner["points"])

In [0]:
import json
from datetime import datetime, timezone

# Reuse the race details extracted from the JSON response.
race = payload["MRData"]["RaceTable"]["Races"][0]

bronze_record = [
    (
        "jolpica",                      # data source
        "race_results",                 # type of data collected
        int(race["season"]),            # e.g. 2026
        int(race["round"]),             # e.g. 17
        json.dumps(payload),            # complete untouched JSON response
        api_url,                        # exact endpoint used
        datetime.now(timezone.utc)      # collection time in UTC
    )
]

bronze_columns = [
    "source_name",
    "entity_type",
    "season",
    "round",
    "payload_json",
    "source_url",
    "ingested_at"
]

bronze_df = spark.createDataFrame(bronze_record, bronze_columns)

display(bronze_df)

In [0]:
catalog = spark.sql("SELECT current_catalog() AS catalog").first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

(
    bronze_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table)
)

print(f"Saved raw response to: {bronze_table}")

In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

bronze_df = spark.table(bronze_table)

display(
    bronze_df
    .select(
        "source_name",
        "entity_type",
        "season",
        "round",
        "source_url",
        "ingested_at"
    )
    .orderBy(F.col("ingested_at").desc())
)